In [1]:
import pandas as pd
import numpy as np

CURRENT_YEAR = 2026

# ============================================================
# LOAD FILES
# ============================================================

df = pd.read_csv("../data/drivearabia_car_depreciation_valuation/original/ram.csv")

dep_df = pd.read_csv(
    "../data/drivearabia_car_depreciation_valuation/annual_dep_rate.csv"
)

# ============================================================
# CLEAN DEPRECIATION DATA
# ============================================================

dep_df["make"] = dep_df["make"].astype(str).str.strip().str.upper()
dep_df["model"] = dep_df["model"].astype(str).str.strip().str.upper()

# ============================================================
# RAM MODEL MAPPING
# ============================================================

MODEL_MAPPING = {
    "ram-1500": "1500",
    "ram-1500-classic": "1500 CLASSIC",
    "ram-2500": "2500",
    "ram-3500": "3500",
    "ram-trx": "TRX",
    "ram-promaster": "PROMASTER",
    "ram-promaster-city": "PROMASTER CITY",
}

# ============================================================
# PREPARE DATA
# ============================================================

df["make"] = "RAM"

df["model_name"] = (
    df["model_slug"]
    .map(MODEL_MAPPING)
    .astype(str)
    .str.upper()
)

# ============================================================
# BUILD LOOKUP
# ============================================================

dep_lookup = (
    dep_df.groupby(["make", "model"])["annual_dep_rate"]
    .mean()
    .reset_index()
)

# ============================================================
# MERGE DEPRECIATION RATE
# ============================================================

df = df.merge(
    dep_lookup,
    left_on=["make", "model_name"],
    right_on=["make", "model"],
    how="left"
)

# ============================================================
# CAR AGE
# ============================================================

df["car_age"] = CURRENT_YEAR - df["year"]
df["car_age"] = df["car_age"].clip(lower=0)

# ============================================================
# DEPRECIATED VALUE
# ============================================================

def calc_depreciated_value(row):

    rate = row["annual_dep_rate"]

    if pd.isna(rate):
        return np.nan

    age = row["car_age"]

    if age == 0:
        return round(row["price_avg_aed"], 0)

    return round(
        row["price_avg_aed"] * ((1 - rate) ** age),
        0
    )

df["depreciated_value"] = df.apply(
    calc_depreciated_value,
    axis=1
)

# ============================================================
# VALIDATION
# ============================================================

print("Total Rows:", len(df))
print("Matched Rates:", df["annual_dep_rate"].notna().sum())
print("Missing Rates:", df["annual_dep_rate"].isna().sum())

print("\nUnmatched Models:")
print(
    df.loc[df["annual_dep_rate"].isna(), "model_slug"]
    .unique()
    .tolist()
)

# ============================================================
# SAVE
# ============================================================

OUTPUT_FILE = "../data/drivearabia_car_depreciation_valuation/depreciated/ram_dep.csv"

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"\nSaved: {OUTPUT_FILE}")

Total Rows: 23
Matched Rates: 22
Missing Rates: 1

Unmatched Models:
['ram-1500-trx']

Saved: ../data/drivearabia_car_depreciation_valuation/depreciated/ram_dep.csv
